# Metis — Distill from an API teacher

Train Metis **forever** on text written by a frontier teacher model
reached through your **OmniRoute** gateway. The loop never stops —
the teacher writes, Metis learns, checkpoints save to Drive.

**Works out of the box** — the notebook already points at your permanent
tunnel (`teacher.alhissn.com`) and a working free model. Just run the cells.
Your PC must have the gateway (port 20128) + `start_tunnel.bat` running.

---
## Step 1 — Mount Google Drive
Checkpoints save straight here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Step 2 — Clone repo + install dependencies

In [ ]:
import os, subprocess

REPO = "https://github.com/iamasrakib/Metis.git"
METIS_DIR = "/content/Metis"

if os.path.isdir(METIS_DIR):
    subprocess.run(['git', '-C', METIS_DIR, 'pull', '--quiet'], check=True)
else:
    subprocess.run(['git', 'clone', '--quiet', REPO, METIS_DIR], check=True)
os.chdir(METIS_DIR)

subprocess.run(['pip', 'install', '-q', 'numpy', 'tqdm', 'tiktoken', 'tokenizers'], check=True)

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'
print(f'Done. GPU: {gpu}')

---
## Step 3 — Teacher credentials

**No secrets required** — the notebook defaults to your permanent tunnel
and the working free model. If you ever need to override, click the
**key icon** (left sidebar) → **+ New secret** (toggle Notebook access ON):

| Name | Value |
|------|-------|
| `TEACHER_URL` | `https://teacher.alhissn.com/v1` |
| `TEACHER_MODEL` | `oc/mimo-v2.5-free` |
| `TEACHER_API_KEY` | any non-empty string |

Then run the cell below.

In [ ]:
import os
from google.colab import userdata

# Defaults already point at your permanent tunnel + a working free model,
# so you can run this notebook with no setup. Set secrets to override.
def _secret(name, default=""):
    try:
        return userdata.get(name)
    except Exception:
        return default

os.environ["METIS_TEACHER_BASE_URL"] = _secret("TEACHER_URL", "https://teacher.alhissn.com/v1")
os.environ["METIS_TEACHER_MODEL"]    = _secret("TEACHER_MODEL", "oc/mimo-v2.5-free")
os.environ["METIS_TEACHER_API_KEY"]  = _secret("TEACHER_API_KEY", "local-gateway")

print(f"Teacher: {os.environ['METIS_TEACHER_MODEL']} @ {os.environ['METIS_TEACHER_BASE_URL']}")

---
## Step 4 — Test teacher + start distilling

Run the cell below — it tests the connection, then starts the infinite loop.

In [ ]:
import os, subprocess

# Test teacher connection
print("Testing teacher connection...")
r = subprocess.run(
    ["python", "-m", "metis.cli", "distill", "--preset", "tiny", "--test-teacher"],
    capture_output=True, text=True
)
print(r.stdout or r.stderr)
if r.returncode != 0:
    print("Teacher check failed. Verify your TEACHER_URL secret and tunnel are running.")
else:
    print()

# Link checkpoints to Drive
DRIVE_BASE = "/content/drive/MyDrive/Metis"
CKPT_DIR   = "checkpoints_distill"
os.makedirs(DRIVE_BASE, exist_ok=True)
drive_ckpt = os.path.join(DRIVE_BASE, CKPT_DIR)
os.makedirs(drive_ckpt, exist_ok=True)
local_link = os.path.abspath(CKPT_DIR)
if not os.path.lexists(local_link):
    os.symlink(drive_ckpt, local_link)
    print(f"Linked -> {drive_ckpt}")

# Train forever. Ctrl+C to stop (saves first). Re-run to resume.
# --max-tokens 256: KEEP IT LOW. The tunnel is fronted by Cloudflare,
# which drops requests after 100s; the 1024-token default hangs forever.
print("Starting distillation...")
result = subprocess.run(
    ["python", "-m", "metis.cli", "distill",
     "--checkpoint-dir", "checkpoints_distill",
     "--preset", "tiny", "--tokenizer", "cl100k_base",
     "--max-tokens", "256", "--min-sleep", "10"],
    capture_output=False
)
if result.returncode != 0:
    print(f"Distillation exited with error code {result.returncode}")

---
**Stop:** `Ctrl+C` in the run cell, or create a file `checkpoints_distill/STOP`.
**Resume:** re-run the cell — it picks up from the last checkpoint.
**Change topic:** add `--topic "animals"` or `--topic-file topics.txt`.
**Limit spend:** add `--budget-tokens 50000` to stop after a token budget.
**Switch model:** change the `TEACHER_MODEL` secret to another available model.